In [ ]:
import trimesh
import pyvista as pv
print(pv.__version__) # Check PyVista version
import numpy as np
import matplotlib.pyplot as plt
import os
# VTK imports
from vtkmodules.vtkFiltersSources import vtkLineSource
from vtkmodules.vtkRenderingCore import (
    vtkPolyDataMapper2D,
    vtkActor2D,
    vtkTextActor,
    vtkCoordinate
)
# Assuming your cross-section functions are importable
# from calculate_cross_section_clean import get_section_points_at_plane # Hypothetical function
# from cross_section_functions import order_points
# --- ICP Alignment (Optional but Recommended) ---
try:
    from trimesh.registration import icp
    use_icp = True
except ImportError:
    print("Warning: 'open3d' or 'pycpd' not found. ICP alignment will be skipped.")
    use_icp = False
# --- End ICP ---


def compare_meshes(original_obj_path, idealised_ply_path, output_dir, base_name):
    """Generates comparison figures for original and idealised meshes, displayed side-by-side."""

    print(f"Comparing '{original_obj_path}' and '{idealised_ply_path}'")
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    try:
        # 1. Load Meshes
        original_mesh = trimesh.load_mesh(original_obj_path)
        idealised_mesh = trimesh.load_mesh(idealised_ply_path)

        if isinstance(original_mesh, trimesh.Scene):
            original_mesh = original_mesh.dump(concatenate=True)
        if isinstance(idealised_mesh, trimesh.Scene):
            idealised_mesh = idealised_mesh.dump(concatenate=True)

        # --- Mesh Integrity Checks ---
        print(f"Original Mesh - Watertight: {original_mesh.is_watertight}")
        print(f"Idealised Mesh - Watertight: {idealised_mesh.is_watertight}")
        # Optional: Try processing to fix minor issues
        # original_mesh.process()
        # idealised_mesh.process()
        # --- End Checks ---

        # 2. Align Meshes
        # Start with centroid alignment
        center_orig = original_mesh.centroid
        center_ideal = idealised_mesh.centroid
        translation = center_orig - center_ideal
        if np.linalg.norm(translation) > 1e-4: # Only translate if significantly different
            print(f"  Aligning idealised mesh centroid to original ({translation})...")
            idealised_mesh.apply_translation(translation)

        # --- Advanced Alignment using ICP (Optional) ---
        if use_icp:
            try:
                # Sample points from both meshes for efficiency
                points_orig, _ = trimesh.sample.sample_surface(original_mesh, 2000)
                points_ideal, _ = trimesh.sample.sample_surface(idealised_mesh, 2000)

                # Run ICP
                icp_result = icp(points_ideal, points_orig, scale=False, max_iterations=100)

                transform_icp = None
                cost = None

                # Check the number of returned values and unpack carefully
                if isinstance(icp_result, (list, tuple)):
                    if len(icp_result) == 3:
                        transform_icp, cost, _ = icp_result
                    elif len(icp_result) == 2:
                        transform_icp, cost = icp_result
                    elif len(icp_result) == 1: # Maybe only transform is returned
                         transform_icp = icp_result[0]
                         cost = -1.0 # Indicate unknown cost
                    else:
                        print("  Warning: ICP returned an unexpected number of values.")
                        cost = -2.0 # Indicate error
                elif isinstance(icp_result, np.ndarray) and icp_result.shape == (4, 4):
                    # Handle case where only the transform matrix is returned
                    print("  Warning: ICP might have returned only the transformation matrix.")
                    transform_icp = icp_result
                    cost = -1.0 # Indicate unknown cost
                else:
                     print("  Warning: ICP returned an unknown format.")
                     cost = -3.0 # Indicate error

                # Validate transform_icp before proceeding
                if transform_icp is None or not isinstance(transform_icp, np.ndarray) or transform_icp.shape != (4, 4):
                    raise ValueError("Failed to retrieve a valid 4x4 transformation matrix from ICP.")

                # Process cost: Ensure it's a float before printing/using
                final_cost_str = "N/A"
                if cost is not None:
                    try:
                        # Attempt to convert cost to float, handle arrays
                        if isinstance(cost, np.ndarray):
                            if cost.size == 1:
                                cost_float = float(cost.item())
                            else:
                                # If cost is an array with multiple values, maybe take the mean or first?
                                print(f"  Warning: ICP cost is an array ({cost}), using first element.")
                                cost_float = float(cost.flat[0])
                        else:
                            cost_float = float(cost)

                        if np.isfinite(cost_float):
                             final_cost_str = f"{cost_float:.6f}"
                        else:
                             final_cost_str = "Invalid (non-finite)"
                    except (TypeError, ValueError) as cost_err:
                        print(f"  Warning: Could not interpret ICP cost value ({cost}): {cost_err}")
                        final_cost_str = f"Error ({cost})"

                print(f"  ICP alignment cost: {final_cost_str}")

                # Apply the found transformation to the idealised mesh
                idealised_mesh.apply_transform(transform_icp)
                print("  Applied ICP transformation to idealised mesh.")

            except Exception as e:
                # Print traceback for unexpected errors during ICP itself
                import traceback
                print(f"  Warning: Error during ICP alignment: {e}.")
                # traceback.print_exc() # Uncomment for full traceback if needed
                print("  Proceeding without ICP.")
        # --- End ICP Alignment ---


        # Convert to PyVista objects AFTER alignment
        original_pv = pv.wrap(original_mesh)
        idealised_pv = pv.wrap(idealised_mesh)

        # --- Calculate combined bounds AFTER alignment for ruler placement ---
        combined_bounds = np.array(original_pv.bounds)
        combined_bounds[0] = min(combined_bounds[0], idealised_pv.bounds[0])
        combined_bounds[1] = max(combined_bounds[1], idealised_pv.bounds[1])
        combined_bounds[2] = min(combined_bounds[2], idealised_pv.bounds[2])
        combined_bounds[3] = max(combined_bounds[3], idealised_pv.bounds[3])
        combined_bounds[4] = min(combined_bounds[4], idealised_pv.bounds[4])
        combined_bounds[5] = max(combined_bounds[5], idealised_pv.bounds[5])

        center_x = (combined_bounds[0] + combined_bounds[1]) / 2.0
        bottom_y = combined_bounds[2] # Y min
        bottom_z = combined_bounds[4] # Z min
        extent_x = combined_bounds[1] - combined_bounds[0]

        # 3. Side-by-Side Visualization (PyVista)
        views = {'xy': 'top', 'xz': 'side'}
        # Define colors
        original_color = 'grey'
        idealised_color = '#D55E00' # Vermillion (colorblind-friendly)

        for view_plane, view_name in views.items():
            plotter = pv.Plotter(shape=(1, 2), off_screen=True, window_size=[2048, 1024], border=False)

            # --- Define Actors ---
            ruler_length_world = 5.0 # 5 µm
            vertical_offset_factor = 0.3
            text_position_viewport = (0.47, 0.05) # Default viewport text position
            line = None # Initialize line variable
            p0, p1 = None, None # Initialize line endpoints

            if view_plane == 'xy': # Top View Z calculation
                ruler_y = bottom_y - extent_x * vertical_offset_factor
                ruler_z = combined_bounds[5] + 1.0 # Slightly above
                p0 = np.array([center_x - ruler_length_world / 2.0, ruler_y, ruler_z])
                p1 = np.array([center_x + ruler_length_world / 2.0, ruler_y, ruler_z])
                line = pv.Line(p0, p1)
            elif view_plane == 'xz': # Side View Z calculation
                ruler_y = 0
                ruler_z = bottom_z - extent_x * vertical_offset_factor # Below mesh
                p0 = np.array([center_x - ruler_length_world / 2.0, ruler_y, ruler_z])
                p1 = np.array([center_x + ruler_length_world / 2.0, ruler_y, ruler_z])
                line = pv.Line(p0, p1)

            # --- Add Actors and Configure Camera ---
            # Subplot 1
            plotter.subplot(0, 0)
            plotter.add_mesh(original_pv, color=original_color, smooth_shading=False, show_edges=False)
            if line:
                 plotter.add_mesh(line, color='black', line_width=10, pickable=False, culling=False)

            # --- Text Placement (View Dependent) ---
            # --- Text Placement (View Dependent) ---
            if view_plane == 'xy': # Top View: Use viewport text
                plotter.add_text("5 µm", position=text_position_viewport, font_size=16, color='black', font='arial', viewport=True, shadow=True)
            elif view_plane == 'xz': # Side View: ALSO use viewport text, but position it lower 
                side_text_position = (0.47, 0.23)  # Adjust y-coordinate to align with line
                plotter.add_text("5 µm", position=side_text_position, font_size=16, color='black', font='arial', viewport=True, shadow=True)
                # No need for the label_actors block - add_text already sets the font properties

            # Set view AFTER adding actors for this subplot
            if view_plane == 'xy': plotter.view_xy()
            elif view_plane == 'xz': plotter.view_xz()
            plotter.camera.zoom(1.2)
            plotter.reset_camera(bounds=plotter.bounds)

            # Subplot 2
            plotter.subplot(0, 1)
            plotter.add_mesh(idealised_pv, color=idealised_color, smooth_shading=False, show_edges=False)
            if line:
                 plotter.add_mesh(line, color='black', line_width=10, pickable=False, culling=False)

            # --- Text Placement (View Dependent) ---
            if view_plane == 'xy': # Top View: Use viewport text
                plotter.add_text("5 µm", position=text_position_viewport, font_size=16, color='black', font='arial', viewport=True, shadow=True)
            elif view_plane == 'xz': # Side View: ALSO use viewport text, but position it lower 
                side_text_position = (0.47, 0.23)  # Adjust y-coordinate to align with line
                plotter.add_text("5 µm", position=side_text_position, font_size=16, color='black', font='arial', viewport=True, shadow=True)


            # Set view AFTER adding actors for this subplot
            if view_plane == 'xy': plotter.view_xy()
            elif view_plane == 'xz': plotter.view_xz()
            plotter.camera.zoom(1.2)
            plotter.reset_camera(bounds=plotter.bounds)

            # Link views AFTER individual setup
            plotter.link_views()

            # --- Screenshot ---
            output_path = os.path.join(output_dir, f"{base_name}_comparison_{view_name}_sidebyside.png")
            print(f"Saving image to: {output_path}")
            plotter.screenshot(output_path, transparent_background=True)
            plotter.close()



    except Exception as e:
        print(f"  Error processing comparison: {e}")
        import traceback
        traceback.print_exc()

# --- Example Usage ---
# Make sure this cell runs if you intend to execute the comparison
# (In Jupyter, you might run this cell directly)
# if __name__ == '__main__': # This check might not be necessary in a notebook cell
original_file = "Meshes/OBJ/Ac_DA_1_3.obj" # Example
idealised_file_std = "results/full_stomata_Ac_DA_1_3_std.ply" # Example
idealised_file_bulge = "results/full_stomata_Ac_DA_1_3_bulge.ply" # Example
comparison_output_dir = "results/comparisons"

base_name_orig = os.path.splitext(os.path.basename(original_file))[0]

if os.path.exists(original_file) and os.path.exists(idealised_file_std):
    compare_meshes(original_file, idealised_file_std, comparison_output_dir, f"{base_name_orig}_vs_std")

if os.path.exists(original_file) and os.path.exists(idealised_file_bulge):
    compare_meshes(original_file, idealised_file_bulge, comparison_output_dir, f"{base_name_orig}_vs_bulge")


0.44.2
Comparing 'Meshes/OBJ/Ac_DA_1_3.obj' and 'results/full_stomata_Ac_DA_1_3_std.ply'
Original Mesh - Watertight: False
Idealised Mesh - Watertight: False
 [-8.66539928  3.28482578 -6.66842517]
 [-5.32353672  1.21076431 -5.51677948]
 ...
 [18.76231415  3.94724361  0.19043197]
 [ 5.17814589 18.1180207  -4.34870655]
 [-1.78342456  3.11350575 -0.22200396]]), using first element.
  ICP alignment cost: 1.875787
  Applied ICP transformation to idealised mesh.
Saving image to: results/comparisons/Ac_DA_1_3_vs_std_comparison_top_sidebyside.png
Original Mesh - Watertight: False
Idealised Mesh - Watertight: False
 [-8.66539928  3.28482578 -6.66842517]
 [-5.32353672  1.21076431 -5.51677948]
 ...
 [18.76231415  3.94724361  0.19043197]
 [ 5.17814589 18.1180207  -4.34870655]
 [-1.78342456  3.11350575 -0.22200396]]), using first element.
  ICP alignment cost: 1.875787
  Applied ICP transformation to idealised mesh.
Saving image to: results/comparisons/Ac_DA_1_3_vs_std_comparison_top_sidebyside.png